# Notebook 02 - Feature Analysis
**Owner: Person B - Week 1**

Stage 2 Week 1 handoff: load cached activations, retrieve top-activating patches, and label retrieved features with CLIP.

This notebook is orchestration only. Implementation lives in `src/cache.py`, `src/sae.py`, and `src/features.py`. Monosemanticity scores and report gallery figures remain Week 2 work.

## 0. Colab Setup
Prepare the runtime, clone the repo in Colab, and install missing dependencies.

In [ ]:
# Prepare the runtime and clone/mount assets when running on Colab.
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/kgunWf/Sparse-Feature-Circuits-in-ViTs.git"
REPO_DIR = Path("/content/Sparse-Feature-Circuits-in-ViTs")
MOUNT_GOOGLE_DRIVE = True
INSTALL_REQUIREMENTS = True
REQUIRED_IMPORTS = ["torch", "torchvision", "timm", "yaml", "h5py", "vit_prisma", "wordfreq"]

def missing_imports():
    return [name for name in REQUIRED_IMPORTS if importlib.util.find_spec(name) is None]

def install_missing_requirements():
    missing = missing_imports()
    if not missing:
        print("Project requirements already importable.")
        return
    print(f"Installing missing project requirements: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    if MOUNT_GOOGLE_DRIVE:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    if INSTALL_REQUIREMENTS:
        install_missing_requirements()
    print(f"Colab repo ready at {Path.cwd()}")
else:
    if INSTALL_REQUIREMENTS:
        install_missing_requirements()
    else:
        missing = missing_imports()
        if missing:
            print(f"Missing local imports: {missing}")
            print(f"Install with: {sys.executable} -m pip install -r requirements.txt")
    print("Local runtime ready.")


## 1. Parameters
Set paths, cache locations, vocabulary options, and full-run controls.

In [ ]:
# Configure paths, caches, vocabulary, and full-cache run options.
from pathlib import Path

COLAB_CACHE_PATH = "/content/drive/MyDrive/XAI - project files/activations.h5"
LOCAL_CACHE_CANDIDATES = ["outputs/caches/activations.h5", "data/activations.h5", "activations.h5"]
LOCAL_CACHE_SOURCE = next((path for path in LOCAL_CACHE_CANDIDATES if Path(path).exists()), LOCAL_CACHE_CANDIDATES[0])
CACHE_PATH_OVERRIDE = COLAB_CACHE_PATH if globals().get("IN_COLAB", False) else LOCAL_CACHE_SOURCE
COLAB_IMAGE_ZIP = "/content/drive/MyDrive/XAI - project files/imagenet_val_5000.zip"
LOCAL_IMAGE_SOURCE = "data/imagenet_val_5000"  # can also be a local .zip path
IMAGE_ROOT_OVERRIDE = COLAB_IMAGE_ZIP if globals().get("IN_COLAB", False) else LOCAL_IMAGE_SOURCE

COLAB_SAE_ZIP = "/content/drive/MyDrive/XAI - project files/saes.zip"
LOCAL_SAE_CANDIDATES = ["outputs/saes", "outputs/saes.zip", "saes.zip", "data/saes.zip"]
LOCAL_SAE_SOURCE = next((path for path in LOCAL_SAE_CANDIDATES if Path(path).exists()), "outputs/saes")
SAE_ROOT_OVERRIDE = COLAB_SAE_ZIP if globals().get("IN_COLAB", False) else LOCAL_SAE_SOURCE

COLAB_FEATURE_OUTPUT_DIR = "/content/drive/MyDrive/XAI - project files/features"
LOCAL_FEATURE_OUTPUT_DIR = "outputs/features"
FEATURE_OUTPUT_DIR = COLAB_FEATURE_OUTPUT_DIR if globals().get("IN_COLAB", False) else LOCAL_FEATURE_OUTPUT_DIR

TOP_K_PATCHES = 5

RUN_FULL_FEATURE_SCAN = True
FULL_FEATURE_IMAGE_COUNT = None  # None means all images in the cache
TOP_PATCH_BATCH_SIZE = 2
FULL_TOP_PATCHES_CACHE = None  # defaults to FEATURE_OUTPUT_DIR/top_patches_layer{layer}_full.pkl.gz
N_FEATURES_TO_INSPECT = 6

VOCAB_SOURCE = "curated"  # "curated" uses utils.clip_vocab; "wordfreq" uses the old 10k English fallback
CURATED_VOCAB_TIERS = [1, 2, 3, 4]
CURATED_LOAD_TIER4 = True
VOCAB_PATH_OVERRIDE = None  # optional custom vocab file; one concept per line
VOCAB_SIZE = 10_000
EXTRA_CONCEPT_TERMS = [
    "bird", "beak", "wing", "feather", "eye", "head", "leg",
    "grass", "water", "sky", "tree", "branch", "leaf", "rock",
    "white", "pink", "black", "brown", "yellow", "red",
    "stripe", "spot", "edge", "texture", "background",
]

CONTEXT_PATCHES = 2
CROP_SIZE = 128
CLIP_TOP_N = 3
CLIP_CROP_SIZE = 96  # model-space crop around each top patch; CLIP processor still resizes to 224
RUN_FULL_CLIP_LABELS = True  # set False to skip full CLIP labeling
FULL_CLIP_LABELS_CACHE = None  # defaults to FEATURE_OUTPUT_DIR/clip_labels_layer{layer}_{vocab}_crop{CLIP_CROP_SIZE}_full.json
CLIP_LABEL_SAVE_EVERY_FEATURES = 512  # checkpoint full CLIP labels after this many features
CLIP_LABEL_FEATURE_BATCH_SIZE = 4


## 2. Load Cache
Resolve local or Colab assets, then load activation-cache metadata.

In [ ]:
# Resolve local/Colab assets and load activation-cache metadata.
from pathlib import Path
import shutil
from zipfile import ZipFile
import sys

CACHE_PATH_OVERRIDE = globals().get("CACHE_PATH_OVERRIDE")
IMAGE_ROOT_OVERRIDE = globals().get("IMAGE_ROOT_OVERRIDE")
SAE_ROOT_OVERRIDE = globals().get("SAE_ROOT_OVERRIDE")

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
assert (repo_root / "src").exists(), f"Could not find repo root from {Path.cwd()}"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config import get_config
from src.cache import load_image_index, load_layer, load_metadata

def project_path(path):
    path = Path(path)
    return path if path.is_absolute() else repo_root / path

def safe_extract_zip(zip_path, extract_root):
    extract_root = extract_root.resolve()
    extract_root.mkdir(parents=True, exist_ok=True)
    with ZipFile(zip_path) as zip_file:
        for member in zip_file.infolist():
            target = (extract_root / member.filename).resolve()
            try:
                target.relative_to(extract_root)
            except ValueError as exc:
                raise ValueError(f"Unsafe path in zip: {member.filename}") from exc
        zip_file.extractall(extract_root)

def ready_sae_layers(sae_root):
    sae_root = Path(sae_root)
    return sorted(
        path.name
        for path in sae_root.glob("layer_*")
        if (path / "weights.pt").exists() and (path / "config.json").exists()
    )

def install_sae_tree(source_root, target_root):
    source_root = Path(source_root)
    target_root = Path(target_root)
    installed = 0
    weight_files = list(source_root.rglob("weights.pt")) + list(source_root.rglob("sae_weights.pt"))
    for weights_path in weight_files:
        config_path = weights_path.with_name("config.json")
        if not config_path.exists():
            alt_config_path = weights_path.with_name("cfg.json")
            config_path = alt_config_path if alt_config_path.exists() else config_path
        if not config_path.exists():
            continue
        layer_name = next((part for part in reversed(weights_path.parts) if part.startswith("layer_")), weights_path.parent.name)
        layer_dir = target_root / layer_name
        layer_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(weights_path, layer_dir / "weights.pt")
        shutil.copy2(config_path, layer_dir / "config.json")
        installed += 1
    return installed

def prepare_sae_root(sae_root_override):
    target_root = repo_root / "outputs" / "saes"
    target_root.mkdir(parents=True, exist_ok=True)
    if sae_root_override is not None:
        sae_source = project_path(sae_root_override)
        if sae_source.suffix.lower() == ".zip":
            assert sae_source.exists(), f"SAE zip not found: {sae_source}"
            extract_root = target_root / f".{sae_source.stem}_extracted"
            marker = extract_root / ".unzipped"
            install_marker = target_root / f".installed_{sae_source.stem}"
            if not marker.exists():
                print(f"Unzipping SAEs to {extract_root} ...")
                safe_extract_zip(sae_source, extract_root)
                marker.touch()
            if not install_marker.exists() or not ready_sae_layers(target_root):
                installed = install_sae_tree(extract_root, target_root)
                if installed:
                    print(f"Installed {installed} SAE layer folders into {target_root}")
                    install_marker.touch()
        elif sae_source.exists() and sae_source.resolve() != target_root.resolve():
            installed = install_sae_tree(sae_source, target_root)
            if installed:
                print(f"Installed {installed} SAE layer folders into {target_root}")
    layers = ready_sae_layers(target_root)
    assert layers, f"No SAE weights found in {target_root}. Set SAE_ROOT_OVERRIDE to an SAE folder or zip."
    return target_root

def prepare_image_root(image_root_override):
    if image_root_override is None:
        return None
    image_root = project_path(image_root_override)
    if image_root.suffix.lower() != ".zip":
        return image_root
    assert image_root.exists(), f"Image zip not found: {image_root}"
    extract_root = repo_root / "data" / image_root.stem
    marker = extract_root / ".unzipped"
    if not marker.exists():
        print(f"Unzipping images to {extract_root} ...")
        safe_extract_zip(image_root, extract_root)
        marker.touch()
    return extract_root

IMAGE_ROOT = prepare_image_root(IMAGE_ROOT_OVERRIDE)
SAE_ROOT = prepare_sae_root(SAE_ROOT_OVERRIDE)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
IMAGE_FILES_BY_NAME = {}
if IMAGE_ROOT is not None and IMAGE_ROOT.exists():
    IMAGE_FILES_BY_NAME = {
        path.name: path
        for path in IMAGE_ROOT.rglob("*")
        if path.suffix.lower() in IMAGE_EXTENSIONS
    }

def image_path(path):
    path_name = Path(str(path).replace("\\", "/")).name
    path = Path(path)
    resolved = project_path(path)
    if resolved.exists() or IMAGE_ROOT is None:
        return resolved
    by_name = IMAGE_FILES_BY_NAME.get(path_name)
    if by_name is not None:
        return by_name
    fallback = IMAGE_ROOT / path
    return fallback if fallback.exists() else IMAGE_ROOT / path_name

def resolve_top_patch_image_paths(top_patches_by_feature):
    changed = False
    for patches in top_patches_by_feature.values():
        for patch in patches:
            old_path = patch.get("image_path")
            if old_path is None:
                continue
            new_path = str(image_path(old_path))
            if old_path != new_path:
                patch["image_path"] = new_path
                changed = True
    return changed

cfg = get_config()
layer = cfg.sae.primary_layer
cache_path = project_path(CACHE_PATH_OVERRIDE or cfg.outputs.cache_path)

assert cache_path.exists(), f"Missing activation cache: {cache_path}"

metadata = load_metadata(cache_path)
index = load_image_index(cache_path)

assert layer in metadata["layers"], f"Layer {layer} not found in cache layers {metadata['layers']}"
assert len(index["paths"]) == metadata["n_images"]

print(f"Cache: {cache_path}")
if IMAGE_ROOT is not None:
    print(f"Image root: {IMAGE_ROOT} ({len(IMAGE_FILES_BY_NAME)} files indexed)")
print(f"SAE root: {SAE_ROOT} ({', '.join(ready_sae_layers(SAE_ROOT))})")
print(f"Model: {metadata['model_name']}")
print(f"Layer: {layer}")
print(f"Images: {metadata['n_images']}")
print(f"First image: {index['paths'][0]}")


## 3. Patch Gallery Helper
Define the compact grid renderer used to inspect full-cache feature patches.

In [ ]:
# Define the patch-grid helper used by the full-cache inspection cells.
from PIL import Image, ImageDraw
from src.features import crop_patch_images


def make_patch_grid(feature_patches, labels=None, context_patches=CONTEXT_PATCHES, crop_size=CROP_SIZE, max_patches=None):
    labels = labels or {}
    max_patches = max_patches or TOP_K_PATCHES
    feature_items = list(feature_patches.items())
    if not feature_items:
        return Image.new("RGB", (320, 80), color="white")

    label_width = 260
    row_height = crop_size + 42
    width = label_width + max_patches * crop_size
    height = row_height * len(feature_items)
    grid = Image.new("RGB", (width, height), color="white")
    draw = ImageDraw.Draw(grid)

    for row, (feature_idx, patches) in enumerate(feature_items):
        y = row * row_height
        title = f"feature {feature_idx}"
        if feature_idx in labels:
            title += " | " + ", ".join(labels[feature_idx])
        draw.text((8, y + 8), title, fill="black")

        acts = [f"{p['activation_value']:.2f}" for p in patches[:max_patches]]
        draw.text((8, y + 28), "acts: " + ", ".join(acts), fill="black")

        crops = crop_patch_images(patches[:max_patches], context_patches=context_patches, mark_patch=True)
        for col, crop in enumerate(crops):
            crop = crop.resize((crop_size, crop_size))
            grid.paste(crop, (label_width + col * crop_size, y))

    return grid


## 4. Full-Cache Top Patches
Load or compute top-activating patches for every SAE feature.

In [ ]:
# Load or compute top patches for every SAE feature.
import gzip
import pickle

from src.features import get_top_patches_all_features_from_cache

feature_output_dir = project_path(globals().get("FEATURE_OUTPUT_DIR", "outputs/features"))
feature_output_dir.mkdir(parents=True, exist_ok=True)
full_top_patches_cache = (
    project_path(FULL_TOP_PATCHES_CACHE)
    if FULL_TOP_PATCHES_CACHE
    else feature_output_dir / f"top_patches_layer{layer}_full.pkl.gz"
)

if RUN_FULL_FEATURE_SCAN:
    full_image_count = metadata["n_images"] if FULL_FEATURE_IMAGE_COUNT is None else min(FULL_FEATURE_IMAGE_COUNT, metadata["n_images"])
    full_image_paths = [str(image_path(index["paths"][i])) for i in range(full_image_count)]
    missing_full_images = [path for path in full_image_paths if not Path(path).exists()]
    assert not missing_full_images, f"Missing image files. Set IMAGE_ROOT_OVERRIDE; first missing: {missing_full_images[0]}"

    if full_top_patches_cache.exists():
        with gzip.open(full_top_patches_cache, "rb") as f:
            all_top_patches = pickle.load(f)
        print(f"Loaded cached full top patches: {full_top_patches_cache}")
    else:
        all_top_patches = get_top_patches_all_features_from_cache(
            layer=layer,
            image_paths=full_image_paths,
            cachepath=cache_path,
            k=TOP_K_PATCHES,
            batch_size=TOP_PATCH_BATCH_SIZE,
            image_count=full_image_count,
        )
        with gzip.open(full_top_patches_cache, "wb") as f:
            pickle.dump(all_top_patches, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Saved full top patches: {full_top_patches_cache}")

    if resolve_top_patch_image_paths(all_top_patches):
        print("Resolved cached top-patch image paths for this runtime.")

    features_to_inspect = sorted(
        all_top_patches,
        key=lambda feature_idx: all_top_patches[feature_idx][0]["activation_value"],
        reverse=True,
    )[:N_FEATURES_TO_INSPECT]
    patches_for_labeling = {
        feature_idx: all_top_patches[feature_idx]
        for feature_idx in features_to_inspect
    }
else:
    raise ValueError("RUN_FULL_FEATURE_SCAN must be True for the full-cache notebook path.")

print(f"Full-cache top-patch results: {len(all_top_patches)} features")
print(f"Features selected for inspection: {features_to_inspect}")
make_patch_grid(patches_for_labeling)


## 5. CLIP Auto-Labeling
Label full-cache features with local crops and the configured concept vocabulary.

In [ ]:
# Build the concept vocab and checkpoint CLIP labels for the selected features.
import json

from src.features import load_clip_labeler, label_features_clip, load_concept_vocab
from utils.clip_vocab import get_vocab

vocab_cache_tag = "custom" if VOCAB_PATH_OVERRIDE else str(VOCAB_SOURCE).lower()
if VOCAB_PATH_OVERRIDE:
    CONCEPT_VOCAB = load_concept_vocab(
        path=VOCAB_PATH_OVERRIDE,
        size=VOCAB_SIZE,
        extra_terms=EXTRA_CONCEPT_TERMS,
    )
elif str(VOCAB_SOURCE).lower() == "curated":
    CONCEPT_VOCAB = get_vocab(tiers=CURATED_VOCAB_TIERS, load_tier4=CURATED_LOAD_TIER4)
elif str(VOCAB_SOURCE).lower() == "wordfreq":
    CONCEPT_VOCAB = load_concept_vocab(
        path=None,
        size=VOCAB_SIZE,
        extra_terms=EXTRA_CONCEPT_TERMS,
    )
else:
    raise ValueError("VOCAB_SOURCE must be 'curated' or 'wordfreq'.")
print(f"Concept vocab size: {len(CONCEPT_VOCAB):,}")

feature_output_dir = project_path(globals().get("FEATURE_OUTPUT_DIR", "outputs/features"))
feature_output_dir.mkdir(parents=True, exist_ok=True)
full_clip_labels_cache = (
    project_path(FULL_CLIP_LABELS_CACHE)
    if FULL_CLIP_LABELS_CACHE
    else feature_output_dir / f"clip_labels_layer{layer}_{vocab_cache_tag}_crop{CLIP_CROP_SIZE}_full.json"
)

label_all_features = RUN_FULL_CLIP_LABELS and bool(all_top_patches)
features_for_clip_labels = all_top_patches if label_all_features else patches_for_labeling
all_labels = {}

if label_all_features and full_clip_labels_cache.exists():
    with open(full_clip_labels_cache, "r", encoding="utf-8") as f:
        cached_labels = json.load(f)
    all_labels = {int(feature_idx): feature_labels for feature_idx, feature_labels in cached_labels.items()}

missing_labels = [feature_idx for feature_idx in features_for_clip_labels if feature_idx not in all_labels]

if label_all_features:
    if missing_labels:
        print(f"CLIP labels missing {len(missing_labels):,} / {len(features_for_clip_labels):,} features; resuming.")
        clip_model, processor = load_clip_labeler()
        save_every = max(1, int(CLIP_LABEL_SAVE_EVERY_FEATURES))
        for start in range(0, len(missing_labels), save_every):
            chunk_ids = missing_labels[start:start + save_every]
            chunk_labels = label_features_clip(
                {feature_idx: features_for_clip_labels[feature_idx] for feature_idx in chunk_ids},
                CONCEPT_VOCAB,
                clip_model,
                processor,
                top_n=CLIP_TOP_N,
                clip_image_size=CLIP_CROP_SIZE,
                feature_batch_size=CLIP_LABEL_FEATURE_BATCH_SIZE,
            )
            all_labels.update(chunk_labels)
            with open(full_clip_labels_cache, "w", encoding="utf-8") as f:
                json.dump({str(feature_idx): feature_labels for feature_idx, feature_labels in sorted(all_labels.items())}, f, indent=2)
            print(f"Saved CLIP label checkpoint: {len(all_labels):,} / {len(features_for_clip_labels):,} features -> {full_clip_labels_cache}")
    else:
        print(f"Loaded cached full CLIP labels: {full_clip_labels_cache}")
else:
    clip_model, processor = load_clip_labeler()
    all_labels = label_features_clip(
        features_for_clip_labels,
        CONCEPT_VOCAB,
        clip_model,
        processor,
        top_n=CLIP_TOP_N,
        clip_image_size=CLIP_CROP_SIZE,
        feature_batch_size=CLIP_LABEL_FEATURE_BATCH_SIZE,
    )

labels = {feature_idx: all_labels.get(feature_idx, []) for feature_idx in patches_for_labeling}
print(f"CLIP-labeled features: {len(all_labels):,}")

for feature_idx, feature_labels in labels.items():
    print(f"feature {feature_idx}: {feature_labels}")

make_patch_grid(patches_for_labeling, labels=labels)


## 6. Week 2 Placeholders
Monosemanticity scores, feature-gallery report figures, and top-50 manual annotation belong to Week 2. Keep those implementations in `src/features.py` and `src/visualise.py`, then call them here once they exist.

## Week 1 Checkpoint
- [x] Activation cache metadata loads
- [ ] Full-cache streaming top patches run over all cached images and save to `FEATURE_OUTPUT_DIR`
- [x] CLIP labels use local crops centered on top patches, resized by the CLIP processor
- [x] CLIP text labels use a curated visual concept vocabulary
- [ ] Full-feature CLIP labels run and save to `FEATURE_OUTPUT_DIR`
- [ ] Manually inspect several feature rows: red boxes should focus on a consistent visual pattern, and CLIP labels should roughly match that pattern